In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/super-resolution-in-video-games-2/sample_submission.csv
/kaggle/input/competitions/super-resolution-in-video-games-2/test/lr/01301.png
/kaggle/input/competitions/super-resolution-in-video-games-2/test/lr/02578.png
/kaggle/input/competitions/super-resolution-in-video-games-2/test/lr/00929.png
/kaggle/input/competitions/super-resolution-in-video-games-2/test/lr/02630.png
/kaggle/input/competitions/super-resolution-in-video-games-2/test/lr/01883.png
/kaggle/input/competitions/super-resolution-in-video-games-2/test/lr/00704.png
/kaggle/input/competitions/super-resolution-in-video-games-2/test/lr/03163.png
/kaggle/input/competitions/super-resolution-in-video-games-2/test/lr/01124.png
/kaggle/input/competitions/super-resolution-in-video-games-2/test/lr/01570.png
/kaggle/input/competitions/super-resolution-in-video-games-2/test/lr/03092.png
/kaggle/input/competitions/super-resolution-in-video-games-2/test/lr/02640.png
/kaggle/input/competitions/super-resolution-in-v

In [8]:
import os
import random
from PIL import Image, UnidentifiedImageError
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
import torchvision.transforms.functional as TF

# --- Dataset Paths ---
DATASET_PATH = "/kaggle/input/competitions/super-resolution-in-video-games-2/"
TRAIN_LR_DIR = os.path.join(DATASET_PATH, "train/lr")
TRAIN_HR_DIR = os.path.join(DATASET_PATH, "train/hr")
TEST_LR_DIR = os.path.join(DATASET_PATH, "test/lr")

PROCESSED_TEST_LR_DIR = "/kaggle/working/test_lr_rgb"

# --- 1. Data Ingestion & Validation (FR-1) ---
def validate_and_split_data(lr_dir, hr_dir, val_split=0.1, seed=42):
    valid_pairs = []
    lr_files = set(os.listdir(lr_dir))
    hr_files = set(os.listdir(hr_dir))
    
    for filename in lr_files:
        if filename not in hr_files:
            print(f"Missing HR match for: {filename}")
            continue
            
        lr_path = os.path.join(lr_dir, filename)
        hr_path = os.path.join(hr_dir, filename)
        
        try:
            with Image.open(lr_path) as img_lr, Image.open(hr_path) as img_hr:
                img_lr.verify()
                img_hr.verify()
                valid_pairs.append((lr_path, hr_path))
        except (UnidentifiedImageError, OSError):
            print(f"Corrupted image detected and skipped: {filename}")

    generator = torch.Generator().manual_seed(seed)
    train_len = int(len(valid_pairs) * (1 - val_split))
    val_len = len(valid_pairs) - train_len
    
    return random_split(valid_pairs, [train_len, val_len], generator=generator)

def validate_test_data(test_lr_dir):
    for filename in os.listdir(test_lr_dir):
        path = os.path.join(test_lr_dir, filename)
        try:
            with Image.open(path) as img:
                if img.size != (64, 64) or img.mode not in ['RGB', 'RGBA']:
                    print(f"Invalid test image (not 64x64 or RGB): {filename}")
        except (UnidentifiedImageError, OSError):
            print(f"Corrupted test image: {filename}")

def convert_and_save_test_images(input_dir, output_dir):
    """Reads test images, converts them to RGB, and saves them to the working directory."""
    os.makedirs(output_dir, exist_ok=True)
    print(f"Converting test images to RGB and saving to {output_dir}...")
    
    for filename in os.listdir(input_dir):
        input_path = os.path.join(input_dir, filename)
        output_path = os.path.join(output_dir, filename)
        
        try:
            with Image.open(input_path) as img:
                rgb_img = img.convert('RGB')
                
                # Verify size during conversion
                if rgb_img.size != (64, 64):
                    print(f"Warning: {filename} has unexpected size {rgb_img.size}")
                    
                rgb_img.save(output_path, format="PNG")
        except (UnidentifiedImageError, OSError) as e:
            print(f"Failed to process {filename}: {e}")
            
    print("Test set conversion complete!")

# --- 2. Preprocessing & Dataset (FR-2) ---
class VideoGameSRDataset(Dataset):
    def __init__(self, image_pairs, lr_crop_size=64, scale_factor=4, augment=True):
        self.image_pairs = image_pairs
        self.lr_crop_size = lr_crop_size
        self.hr_crop_size = lr_crop_size * scale_factor 
        self.augment = augment

    def __len__(self):
        return len(self.image_pairs)

    def __getitem__(self, idx):
        lr_path, hr_path = self.image_pairs[idx]
        lr_img = Image.open(lr_path).convert('RGB')
        hr_img = Image.open(hr_path).convert('RGB')

        if self.augment:
            # Synchronized Random Crop
            i, j, h, w = transforms.RandomCrop.get_params(lr_img, output_size=(self.lr_crop_size, self.lr_crop_size))
            lr_img = TF.crop(lr_img, i, j, h, w)
            hr_img = TF.crop(hr_img, i * 4, j * 4, h * 4, w * 4) 

            # Identical Horizontal Flip
            if random.random() > 0.5:
                lr_img = TF.hflip(lr_img)
                hr_img = TF.hflip(hr_img)

            # Identical 90-Degree Rotation
            rot_angle = random.choice([0, 90, 180, 270])
            if rot_angle > 0:
                lr_img = TF.rotate(lr_img, rot_angle)
                hr_img = TF.rotate(hr_img, rot_angle)

        lr_tensor = TF.to_tensor(lr_img)
        hr_tensor = TF.to_tensor(hr_img)

        return lr_tensor, hr_tensor

def denormalize(tensor):
    return (tensor.clamp(0, 1) * 255).byte()

# --- Initialization Setup ---
if __name__ == "__main__":
    # 1. Convert and save test images to RGB format in Kaggle's working directory
    convert_and_save_test_images(TEST_LR_DIR, PROCESSED_TEST_LR_DIR)
    # Validate test sets before training
    validate_test_data(TEST_LR_DIR)
    
    # Split training and validation pairs
    train_pairs, val_pairs = validate_and_split_data(TRAIN_LR_DIR, TRAIN_HR_DIR, val_split=0.1)
    
    # Create Datasets
    train_dataset = VideoGameSRDataset(train_pairs, lr_crop_size=64, augment=True)
    val_dataset = VideoGameSRDataset(val_pairs, lr_crop_size=64, augment=False)
    
    # Create DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
    
    print(f"Training batches: {len(train_loader)} | Validation batches: {len(val_loader)}")

Converting test images to RGB and saving to /kaggle/working/test_lr_rgb...
Test set conversion complete!
Training batches: 812 | Validation batches: 91
